In [3]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from  sklearn.model_selection  import train_test_split
from sklearn.decomposition import PCA
from sklearn.decomposition import IncrementalPCA
from sklearn.decomposition import KernelPCA
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline


In [4]:
mnist = fetch_openml("mnist_784", version=1, as_frame=False)
X, y = mnist.data, mnist.target

print(X.shape)  # (70000, 784)
print(y.shape)  # (70000,)

(70000, 784)
(70000,)


# SVD ilə PCA-nı əl ilə hesablamaq

In [5]:
# X_centered = X - X.mean(axis=0)      # Hər feature-un ortasını çıxaraq verilənləri mərkəzləşdiririk
# U, s, Vt = np.linalg.svd(X_centered,) # SVD dekompozisiyası: X = U · Σ · V^T
# c1 = Vt.T[:, 0]                     # 1-ci əsas komponent (ən böyük variasiya istiqaməti)
# c2 = Vt.T[:, 1]                     # 2-ci əsas komponent (2-ci ən böyük variasiya istiqaməti)
# W2 = Vt.T[:, :2]              # İlk 2 principal component-dən ibarət çevirmə matrisi
# X2D = X_centered.dot(W2)      # Verilənləri 2 ölçüyə endiririk (proyeksiya)


# Scikit-Learn ilə PCA

In [6]:

pca = PCA(n_components=2)
X2D = pca.fit_transform(X)
pca.components_
pca.components_               # Hər sətir bir principal component-dir


array([[-0., -0., -0., ..., -0., -0., -0.],
       [-0., -0., -0., ..., -0., -0., -0.]], shape=(2, 784))

In [7]:
pca.components_.T[:, 0]       # 1-ci əsas komponent vektoru


array([-0.00000000e+00, -0.00000000e+00, -0.00000000e+00, -0.00000000e+00,
       -0.00000000e+00, -0.00000000e+00, -0.00000000e+00, -0.00000000e+00,
       -0.00000000e+00, -0.00000000e+00, -0.00000000e+00, -0.00000000e+00,
       -9.45584907e-07, -3.72373918e-06, -1.83973346e-06, -7.66555608e-08,
       -0.00000000e+00,  5.55111512e-17, -0.00000000e+00, -3.46944695e-18,
        1.30104261e-18, -1.08420217e-19,  2.71050543e-20, -0.00000000e+00,
        1.69406589e-21, -0.00000000e+00, -1.05879118e-22, -6.61744490e-24,
       -0.00000000e+00, -5.16987883e-26, -0.00000000e+00, -0.00000000e+00,
        2.16824828e-07,  4.29492620e-07,  4.87706215e-06,  1.65229866e-05,
        2.20363253e-05,  3.88808469e-05,  7.40607809e-05,  8.83248604e-05,
        8.08496291e-05,  8.95693486e-05,  1.06176520e-04,  7.93597126e-05,
        3.60812098e-05,  3.08090474e-05,  2.39076582e-05,  3.64534622e-06,
       -2.94191043e-06,  1.88455271e-06,  1.14903055e-06,  4.83408762e-07,
       -0.00000000e+00, -

In [8]:
pca.explained_variance_ratio_ # Hər PC boyunca variasiyanın neçə faizi saxlanıldığını göstərir


array([0.09746116, 0.07155445])

In [9]:
X_train,y_train,X_test,y_test = train_test_split(X,y,test_size=0.2, random_state=42)

In [10]:
pca = PCA()                                 # Bütün komponentləri hesabla
pca.fit(X_train)                            # PCA modelini öyrət
cumsum = np.cumsum(pca.explained_variance_ratio_)  # Variasiya cəmlərini topla
d = np.argmax(cumsum >= 0.95) + 1            # 95%-i keçən minimum ölçü sayı


In [11]:
pca = PCA(n_components=0.95)           # Variasiyanın 95%-ni saxlayacaq qədər ölçü seç
X_reduced = pca.fit_transform(X_train)  # Ölçünü avtomatik azaldır


In [12]:
pca = PCA(n_components=154)              # Ölçünü 784 → 154 endir
X_reduced = pca.fit_transform(X_train)   # Sıxılmış təqdimat
X_recovered = pca.inverse_transform(X_reduced)  # Yenidən 784 ölçüyə bərpa et

# Incremental PCA (böyük dataset üçün)

In [13]:
n_batches = 100                                # Dataseti 100 hissəyə böl
inc_pca = IncrementalPCA(n_components=154)     # IPCA modeli

for X_batch in np.array_split(X_train, n_batches):  # Hər mini-batch üçün
    inc_pca.partial_fit(X_batch)                # Modeli hissə-hissə öyrət

X_reduced = inc_pca.transform(X_train)          # Sonda bütün datasetə tətbiq et


# Memmap ilə IPCA (diskdən oxuma)

In [14]:
# X_mm = np.memmap(filename, dtype="float32", mode="readonly", shape=(m, n))  # Diskdəki böyük array
# batch_size = m // n_batches
# inc_pca = IncrementalPCA(n_components=154, batch_size=batch_size)
# inc_pca.fit(X_mm)   # RAM-ı yükləmədən PCA öyrənir


# Randomized PCA

In [15]:
rnd_pca = PCA(n_components=154, svd_solver="randomized")  # Təxmini, amma sürətli PCA
X_reduced = rnd_pca.fit_transform(X_train)


# Kernel PCA (RBF kernel)

In [ ]:
#  Yaddas catismazligindan islemir
rbf_pca = KernelPCA(n_components=2, kernel="rbf", gamma=0.04)  # Qeyri-xətti ölçü azaltma
X_reduced = rbf_pca.fit_transform(X)


#  Ona gore bu  isledirik
idx = np.random.choice(len(X), 5000, replace=False)  # 5k sample seç
X_small = X[idx]

kpca = KernelPCA(n_components=2, kernel="rbf", gamma=0.04)
X_reduced = kpca.fit_transform(X_small)


# kPCA + Logistic Regression pipeline

In [ ]:
clf = Pipeline([
    ("kpca", KernelPCA(n_components=2)),  # Ölçünü azalt
    ("log_reg", LogisticRegression())     # Klassifikasiya et
])

param_grid = [{
    "kpca__gamma": np.linspace(0.03, 0.05, 10),  # RBF parametri
    "kpca__kernel": ["rbf", "sigmoid"]           # Kernel seçimi
}]

grid_search = GridSearchCV(clf, param_grid, cv=3)  # 3-fold CV ilə axtarış
grid_search.fit(X, y)                              # Ən yaxşı parametrləri tap


# Kernel PCA ilə inverse transform (pre-image)

In [ ]:
rbf_pca = KernelPCA(n_components=2, kernel="rbf", gamma=0.0433,
                    fit_inverse_transform=True)  # İnversiya öyrən
X_reduced = rbf_pca.fit_transform(X)
X_preimage = rbf_pca.inverse_transform(X_reduced)  # Orijinal fəzaya təxmini bərpa


# Reconstruction error

In [ ]:
from sklearn.metrics import mean_squared_error

mean_squared_error(X, X_preimage)  # Orijinal və bərpa edilmiş verilənlər arasındakı MSE


# LLE (Locally Linear Embedding)

In [ ]:
from sklearn.manifold import LocallyLinearEmbedding

lle = LocallyLinearEmbedding(n_components=2, n_neighbors=10)  # 10 qonşu ilə manifold öyrən
X_reduced = lle.fit_transform(X)  # Qeyri-xətti şəkildə 2 ölçüyə endir
